In [2]:

# Aim: To implement Autoencoder for Anomaly Detection

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# Step 1: Access the dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Step 2: Preprocessing
x_train = x_train / 255.0
x_test = x_test / 255.0

x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# Use only normal data (digit 0) for training
x_train = x_train[y_train == 0]
x_test_normal = x_test[y_test == 0]
x_test_anomaly = x_test[y_test != 0]

# Step 3: Build Encoder
encoder = models.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu")
])

# Step 4: Build Decoder
decoder = models.Sequential([
    layers.Input(shape=(16,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(784, activation="sigmoid")
])

# Step 5: Build Autoencoder
autoencoder = models.Sequential([
    encoder,
    decoder
])

# Step 6: Compile the model
autoencoder.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# Step 7: Train the model
autoencoder.fit(
    x_train, x_train,
    epochs=5,
    batch_size=256,
    validation_data=(x_test_normal, x_test_normal)
)

# Step 8: Evaluate the model
loss, mae = autoencoder.evaluate(x_test_normal, x_test_normal)

print("Test Loss:", loss)
print("Test MAE:", mae)

# Step 9: Calculate reconstruction error
normal_output = autoencoder.predict(x_test_normal)
anomaly_output = autoencoder.predict(x_test_anomaly)

normal_error = np.mean((x_test_normal - normal_output) ** 2, axis=1)
anomaly_error = np.mean((x_test_anomaly - anomaly_output) ** 2, axis=1)

# Step 10: Set threshold
threshold = np.mean(normal_error) + 2 * np.std(normal_error)

print("Threshold:", threshold)

# Step 11: Detect anomalies
print("Normal Data Error:", np.mean(normal_error))
print("Anomaly Data Error:", np.mean(anomaly_error))

print("Normal Samples:", np.sum(normal_error < threshold))
print("Anomalies Detected:", np.sum(anomaly_error > threshold))


Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - loss: 0.1868 - mae: 0.4118 - val_loss: 0.1048 - val_mae: 0.2699
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0744 - mae: 0.1790 - val_loss: 0.0607 - val_mae: 0.1393
Epoch 3/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0572 - mae: 0.1325 - val_loss: 0.0530 - val_mae: 0.1256
Epoch 4/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0504 - mae: 0.1202 - val_loss: 0.0469 - val_mae: 0.1145
Epoch 5/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0448 - mae: 0.1094 - val_loss: 0.0431 - val_mae: 0.1048
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0431 - mae: 0.1048
Test Loss: 0.043113548308610916
Test MAE: 0.10483205318450928
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Threshold: 0.07441699348204037
Normal Data Error: 0.043113549620457915
Anomaly Data Error: 0.0923440702704622
Normal Samples: 937
Anomalies Detected: 8094
